In [1]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
print("Đang nạp lại mô hình từ Google Drive...")
shutil.copytree('/content/drive/MyDrive/phobert_education_model_final', './phobert_education_model_final', dirs_exist_ok=True)
print("✅ Nạp xong! Đã sẵn sàng chạy Giao diện Web.")

Mounted at /content/drive
Đang nạp lại mô hình từ Google Drive...
✅ Nạp xong! Đã sẵn sàng chạy Giao diện Web.


In [2]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn.functional as F
import joblib
import re
from pyvi import ViTokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Cấu hình giao diện rộng
st.set_page_config(page_title="EduSentiment ABSA", page_icon="📊", layout="wide")

st.title("📊 Phân tích Cảm xúc Đa Khía Cạnh (ABSA)")
st.markdown("Hệ thống so sánh xác suất chi tiết: Baseline (TF-IDF) vs PhoBERT.")

# 1. Tải các mô hình
@st.cache_resource
def load_all_models():
    # THI CHÚ Ý: Đảm bảo đường dẫn này là đường dẫn thực tế trên Drive của bạn
    base_path = "/content/drive/MyDrive/phobert_education_model_final/"

    phobert_path = base_path
    tfidf_path   = base_path + "tfidf_vectorizer.pkl"
    lr_path      = base_path + "lr_model.pkl"
    xgb_path     = base_path + "xgb_model.pkl"

    tokenizer = AutoTokenizer.from_pretrained(phobert_path)
    model = AutoModelForSequenceClassification.from_pretrained(phobert_path)

    tfidf = joblib.load(tfidf_path)
    lr = joblib.load(lr_path)
    xgb = joblib.load(xgb_path)

    return tokenizer, model, tfidf, lr, xgb

tokenizer, model, tfidf, lr_model, xgb_model = load_all_models()

# Định nghĩa các khía cạnh và từ khóa đi kèm
ASPECT_KEYWORDS = {
    "👨‍🏫 Giảng viên": ['giảng viên', 'thầy', 'cô', 'dạy', 'giảng', 'truyền đạt', 'nhiệt tình'],
    "🏢 Cơ sở vật chất": ['phòng', 'máy', 'nóng', 'wifi', 'thiết bị', 'wc', 'vệ sinh', 'bàn ghế', 'điều hòa'],
    "📚 Chương trình học": ['chương trình', 'môn', 'kiến thức', 'thực hành', 'lý thuyết', 'nội dung'],
    "📄 Tài liệu": ['tài liệu', 'slide', 'sách', 'bài tập', 'giáo trình']
}

label_map = {0: 'Tiêu cực 😡', 1: 'Trung tính 😐', 2: 'Tích cực 😍'}

def get_sentiment(text, is_phobert=True):
    if is_phobert:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1).squeeze().tolist()
            pred = torch.argmax(outputs.logits, dim=1).item()
        return pred, probs
    else:
        # Dùng cho mô hình Baseline (TF-IDF)
        segmented = ViTokenizer.tokenize(text)
        features = tfidf.transform([segmented])

        # Dự đoán nhãn
        lr_p = lr_model.predict(features)[0]
        xgb_p = xgb_model.predict(features)[0]

        # Lấy xác suất (predict_proba)
        lr_probs = lr_model.predict_proba(features)[0].tolist()
        xgb_probs = xgb_model.predict_proba(features)[0].tolist()

        return lr_p, lr_probs, xgb_p, xgb_probs

# 2. Nhập liệu
user_input = st.text_area("Nhập ý kiến phản hồi của sinh viên:", height=100,
                          placeholder="Ví dụ: Giảng viên dạy rất hay nhưng phòng máy hơi nóng và wifi yếu.")

if st.button("Phân tích chi tiết"):
    if not user_input.strip():
        st.warning("Vui lòng nhập nội dung!")
    else:
        with st.spinner('Đang tính toán xác suất cho cả 3 mô hình...'):
            # Tách câu theo các dấu ngắt hoặc từ nối "nhưng" để phân tích khía cạnh chính xác hơn
            segments = re.split(r'[,.;!]| nhưng | và ', user_input)

            aspect_results = []

            for segment in segments:
                segment = segment.strip()
                if not segment: continue

                # Kiểm tra đoạn này thuộc khía cạnh nào
                for aspect, keywords in ASPECT_KEYWORDS.items():
                    if any(kw in segment.lower() for kw in keywords):
                        # Dự đoán bằng PhoBERT
                        phobert_idx, phobert_probs = get_sentiment(segment, is_phobert=True)
                        # Dự đoán bằng Baseline
                        lr_idx, lr_probs, xgb_idx, xgb_probs = get_sentiment(segment, is_phobert=False)

                        aspect_results.append({
                            "aspect": aspect,
                            "text": segment,
                            "phobert": label_map[phobert_idx],
                            "phobert_probs": phobert_probs,
                            "lr": label_map[lr_idx],
                            "lr_probs": lr_probs,
                            "xgb": label_map[xgb_idx],
                            "xgb_probs": xgb_probs
                        })
                        break

            # 3. Hiển thị kết quả
            if not aspect_results:
                st.info("Không tìm thấy khía cạnh cụ thể trong câu. Vui lòng nhập rõ hơn (ví dụ nhắc đến giảng viên, phòng máy...)")
            else:
                st.subheader("📊 Bảng so sánh xác suất chi tiết theo khía cạnh")

                labels_name = ["Tiêu cực", "Trung tính", "Tích cực"]

                for res in aspect_results:
                    with st.container():
                        st.markdown(f"### 📌 Khía cạnh: {res['aspect']}")
                        st.caption(f"*Đoạn văn phân tích:* \"{res['text']}\"")

                        # Tạo 3 cột cho 3 mô hình
                        c1, c2, c3 = st.columns(3)

                        # Cột 1: Logistic Regression
                        with c1:
                            st.info(f"**Logistic Regression**\n\nKết quả: **{res['lr']}**")
                            for i in range(3):
                                st.write(f"{labels_name[i]}: {res['lr_probs'][i]*100:.1f}%")
                                st.progress(float(res['lr_probs'][i]))

                        # Cột 2: XGBoost
                        with c2:
                            st.warning(f"**XGBoost**\n\nKết quả: **{res['xgb']}**")
                            for i in range(3):
                                st.write(f"{labels_name[i]}: {res['xgb_probs'][i]*100:.1f}%")
                                st.progress(float(res['xgb_probs'][i]))

                        # Cột 3: PhoBERT
                        with c3:
                            st.success(f"**PhoBERT (Đề xuất)**\n\nKết quả: **{res['phobert']}**")
                            for i in range(3):
                                st.write(f"{labels_name[i]}: {res['phobert_probs'][i]*100:.1f}%")
                                st.progress(float(res['phobert_probs'][i]))

                        st.divider()

Writing app.py


In [3]:
!pip install streamlit pyvi -q
!pkill -f streamlit
!pkill -f cloudflared

import time
import os

if not os.path.exists('cloudflared'):
    print("⬇️ Đang tải lại công cụ tạo đường link (Cloudflare)...")
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

print("⏳ Đang khởi động Streamlit và từ từ nạp AI PhoBERT...")
!nohup streamlit run app.py --server.port 8501 > logs.txt 2>&1 &
time.sleep(15)

print("\n📋 Trạng thái khởi động ứng dụng (Logs):")
!tail -n 10 logs.txt

print("\n🌐 Bật lại đường hầm Cloudflare...")
!nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
time.sleep(8)

print("\n" + "="*70)
print("🎉 LINK MỚI CỦA BẠN ĐÂY (Bấm vào là mở được web ngay):")
print("="*70)
!grep -o 'https://[^[:space:]]*\.trycloudflare\.com' cloudflared.log

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.5 MB/s eta 0:00:00
⬇️ Đang tải lại công cụ tạo đường link (Cloudflare)...
⏳ Đang khởi động Streamlit và từ từ nạp AI PhoBERT...

📋 Trạng thái khởi động ứng dụng (Logs):

2026-05-13 03:59:28.565 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.230.69.81:8501


🌐 Bật lại đường hầm Cloudflare...

🎉 LINK MỚI CỦA BẠN ĐÂY (Bấm vào là mở được web ngay):
https://probe-pixels-instant-suggested.trycloudflare.com
